In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_34.pickle

In [ ]:
%%RecordEvent
%%time
### cell 35 ###

### cell 35 optimized ###

def grab_subset_of_data_47(original_df, question_of_interest):
    # Select relevant columns and rename
    subset_cols = [col for col in original_df.columns if question_of_interest in col]
    mapper = [col.split('-')[-1].lstrip() for col in subset_cols]
    df_subset = original_df[subset_cols].rename(columns=dict(zip(subset_cols, mapper)))
    return df_subset.dropna(how='all')


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
    question_of_interest,
    include_2017=False
):
    # Map each year to its source DataFrame
    response_map = {
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10,
        '2017': responses_df_2017
    }
    years = ['2018','2019','2020','2021','2022']
    if include_2017:
        years.insert(0, '2017')

    dfs = []
    for yr in years:
        df_year = grab_subset_of_data_47(response_map[yr], question_of_interest)
        df_year['year'] = yr
        dfs.append(df_year)

    combined = pd.concat(dfs, ignore_index=True)
    counts = combined.groupby('year').count().reset_index()
    return combined, counts

# Rename 2018 columns to match new question text
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
      .str.replace(
          question_of_interest_old_cell47,
          question_of_interest_new_cell47,
          regex=False
      )
)

# Combine responses from multiple years
ml_df_combined_cell47, ml_df_combined_counts_cell47 = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_47(
        question_of_interest_cell47
    )
)

# Combine related technology columns in one pass
ml_df = ml_df_combined_cell47.copy()
group_defs = [
    ('TensorFlow/Keras', ['TensorFlow','TensorFlow ','Keras','Keras ']),
    ('PyTorch/Lightning/Fast.ai', ['PyTorch','PyTorch ','PyTorch Lightning ','Fast.ai ','Fastai']),
    ('Xgboost/LightGBM/CatBoost', ['Xgboost','Xgboost ','lightgbm','LightGBM ','catboost','CatBoost ']),
    ('Scikit-learn', ['Scikit—learn ','Scikit—learn ','learn ','Learn'])
]

# Build each new column and drop originals at once
to_drop = []
for label, cols in group_defs:
    mask = ml_df[cols].notna().any(axis=1)
    ml_df[label] = label
    ml_df[label] = ml_df[label].where(mask, None)
    to_drop.extend(cols)
ml_df = ml_df.drop(columns=to_drop)

# Recompute counts per year
ml_counts = ml_df.groupby('year').count().reset_index()

# Vectorized percentage conversion
def convert_df_of_counts_to_percentages_47(data_df, counts_df):
    counts_idx = counts_df.set_index('year')
    totals = data_df['year'].value_counts().sort_index()
    pct = counts_idx.div(totals, axis=0) * 100
    return pct.reset_index()

ml_df_percentages = convert_df_of_counts_to_percentages_47(ml_df, ml_counts)

# Prepare for plotting
df_cell47 = (
    ml_df_percentages
      .melt(
          id_vars=['year'],
          value_vars=[
              'Scikit-learn',
              'Xgboost/LightGBM/CatBoost',
              'TensorFlow/Keras',
              'PyTorch/Lightning/Fast.ai'
          ]
      )
      .sort_values(['year','value'])
      .rename(columns={'variable': ''})
)

df_cell47.info()

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_35_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_35_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[35], f)


In [ ]:
opt_output = Out.get(4)